![Week 4 Day 2](../../assets/banners/w4d2.png)

# Week 4 Day 2 - Solution: Citation Verdicts and the Answer Weight

Two parts: give the offline citation checker a `verdict` column, then measure how much the student's answer is allowed to steer `find_evidence()` for question Q8 (DNS).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Try the exercise yourself before reading on: struggling a little is where the learning happens! These are <i>reference</i> solutions, not the only right answers.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json
import re

from langchain_core.embeddings import DeterministicFakeEmbedding

import gradewise.rag.evidence as evidence_module
from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.ingestion import ingest
from gradewise.rag import find_evidence, get_retriever, unique_citations
from gradewise.schemas import ExamPaper

OFFLINE_DB = OUTPUT_DIR / "week4_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
# 📦 ingest() reads datasets.kb_files(): 4 PDFs, or 2 with DATA_SIZE=small
ingest(collection="cs201_offline", embeddings=fake_embeddings, persist_dir=OFFLINE_DB, quiet=True)
bm25 = get_retriever("cs201_offline", k=5, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

### Part 1: a `verdict` column

In [ ]:
CITE_RE = re.compile(r"\[([^\[\]]+)\]")
STOP = {"the", "a", "an", "of", "and", "or", "is", "are", "to", "in", "for", "it", "that", "can", "be", "by", "at"}


def content_words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOP and len(w) > 2}


def verdict(cites: int, unknown: list[str], support: float) -> str:
    if unknown:
        return "🧟 invented source"
    if not cites:
        return "❔ uncited"
    if support < 0.5:
        return "🎭 weak support"
    return "✅ ok"


def check_citations(answer: str, docs) -> list[dict]:
    by_cite = {}
    for doc in docs:
        by_cite.setdefault(unique_citations([doc])[0], []).append(doc.page_content)
    rows = []
    for sentence in re.split(r"(?<=[.!?])\s+", answer.strip()):
        cites = CITE_RE.findall(sentence)
        claim = CITE_RE.sub("", sentence).replace(" .", ".")
        known = [c for c in cites if c in by_cite]
        unknown = [c for c in cites if c not in by_cite]
        evidence = content_words(" ".join(t for c in known for t in by_cite[c]))
        claim_words = content_words(claim)
        support = len(claim_words & evidence) / len(claim_words) if claim_words and known else 0.0
        rows.append(
            {
                "sentence": claim.strip()[:70],
                "support": round(support, 2),
                "verdict": verdict(len(cites), unknown, support),
            }
        )
    return rows


docs = bm25.invoke("four necessary conditions for deadlock")
deadlocks = next(c for c in unique_citations(docs) if "Deadlocks" in c)
mixed = (
    f"A deadlock can arise only if four conditions hold: mutual exclusion, hold and wait, no preemption and "
    f"circular wait [{deadlocks}]. Deadlocks are fixed by rebooting the computer [{deadlocks}]. "
    "They were first described by quantum schedulers [unit9_quantum.pdf p2]. Processes wait for each other."
)
show_table(check_citations(mixed, docs))

💡 Because the checker works **sentence by sentence**, one answer can get four different verdicts. That's what you want in production: keep the good sentences, regenerate or flag the rest. The 0.5 threshold is a judgement call; tune it on real answers, and remember that word overlap is only the cheap first filter before the LLM judge.

### Part 2: how much should the student's answer steer the evidence?

`find_evidence()` reads the module-level `ANSWER_WEIGHT` every time it runs, so we can change it for an experiment (and put it back afterwards).

In [ ]:
exam = ExamPaper.model_validate_json((DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text("utf-8"))
q8 = next(q for q in exam.questions if q.id == "Q8")
truth_file = DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json"
weak = (
    json.loads(truth_file.read_text(encoding="utf-8"))["student_07"]["Q8"]
    if truth_file.exists()
    else "DNS is the domain name system, it finds the IP adress of a website."
)
print(q8.text, "\n\nweak answer:", weak)

default = evidence_module.ANSWER_WEIGHT
columns = {}
try:
    for weight in (0.0, 0.3, 1.0, 5.0):
        evidence_module.ANSWER_WEIGHT = weight
        columns[f"weight {weight}"] = [d.metadata["citation"] for d in find_evidence(q8, weak, k=4, retriever=bm25)]
finally:
    evidence_module.ANSWER_WEIGHT = default  # ⚠️ always restore module state after an experiment
show_table([{"rank": i + 1, **{name: cites[i] for name, cites in columns.items()}} for i in range(4)])

At **0** and **0.3** the evidence is driven by the examiner's queries (the question, its reference topics and every criterion). As the weight grows towards the weight of a whole examiner query (1.0) and beyond, the student's own words start to reorder the list, and at **5.0** the answer dominates: whatever the student happened to mention gets pulled into the "evidence". GradeWise's 0.3 is deliberately small: enough for a correct-but-unexpected point to surface its passage, never enough to let a wrong answer pick its own evidence.